In [6]:
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import Point

In [28]:
df = pd.read_csv("argenprop_1775261078_geo.tsv", sep="\t", encoding="utf-8-sig")

In [29]:
df.dropna(subset="Latitud",inplace=True)
df.drop(columns=["Descripción", "Detalles","Procesada","Toilettes","Tipo_Balcon","Estado_Edificio","Deptos_Por_Piso","Cant_Pisos_Edificio","Expensas_Ficha","Precio_Ficha"],inplace=True)

In [30]:
df.isna().sum().sort_values(ascending=False)

Piso                             2215
Tipo_Unidad                      1847
Estado                           1458
Antiguedad                       1142
Expensas                         1072
Dormitorios                       670
Disposicion                       501
Sup_Cubierta_m2                   239
Baños                             142
Sup_Total_m2                      103
Ambientes                          52
Calle                               0
Precio                              0
Link                                0
Altura                              0
Aire_acondicionado_individual       0
Losa_radiante                       0
Gas_natural                         0
Agua_corriente                      0
Balcón                              0
Terraza                             0
Jardín                              0
Patio                               0
Baulera                             0
Cochera                             0
Muebles_de_cocina                   0
Lavarropas  

In [ ]:
# ─── CARGAR DF ────────────────────────────────────────────────────────────────

df = pd.read_csv("Argenprop_Lat_Lon.tsv", sep="\t", encoding="utf-8-sig")
df = df.dropna(subset=["Latitud", "Longitud"])

gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["Longitud"], df["Latitud"]),
    crs="EPSG:4326"
).to_crs("EPSG:22185")

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def distancia_al_mas_cercano(gdf_props, gdf_pois):
    gdf_pois = gdf_pois.to_crs("EPSG:22185")
    return gdf_props.geometry.apply(lambda geom: gdf_pois.distance(geom).min())

def nombre_mas_cercano(gdf_props, gdf_pois, name_col):
    gdf_pois = gdf_pois.to_crs("EPSG:22185")
    def get_name(geom):
        idx = gdf_pois.distance(geom).idxmin()
        return gdf_pois.loc[idx, name_col]
    return gdf_props.geometry.apply(get_name)

def count_within(gdf_props, gdf_pois, radius=300):
    # FIX: verificar que gdf_pois no esté vacío antes de calcular
    if len(gdf_pois) == 0:
        return pd.Series(0, index=gdf_props.index)
    gdf_pois = gdf_pois.to_crs("EPSG:22185")
    return gdf_props.geometry.apply(lambda geom: (gdf_pois.distance(geom) <= radius).sum())

# ─── DESCARGAR GCBA ───────────────────────────────────────────────────────────

print("📥 Descargando datasets del GCBA...")
GCBA = {
    "barrios":       "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/ministerio-de-educacion/barrios/barrios.geojson",
    "subtes":        "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-estaciones/estaciones-de-subte.geojson",
    "hospitales":    "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/ministerio-de-salud/hospitales/hospitales.geojson",
    "universidades": "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/ministerio-de-educacion/universidades/universidades.geojson",
}

gdfs = {}
for nombre, url in GCBA.items():
    try:
        gdfs[nombre] = gpd.read_file(url)
        print(f"   ✅ {nombre}")
    except Exception as e:
        print(f"   ❌ {nombre}: {e}")

# ─── BARRIO Y COMUNA ──────────────────────────────────────────────────────────

print("\n🏘️  Barrio y comuna...")

# FIX: limpiar columnas duplicadas antes del sjoin para evitar conflictos
cols_a_limpiar = ["nombre", "Barrio", "commune", "Comuna", "index_right"]
for col in cols_a_limpiar:
    if col in gdf.columns:
        gdf = gdf.drop(columns=[col])

gdf_barrios = gdfs["barrios"].to_crs("EPSG:22185")

# FIX: solo traer las columnas necesarias del GeoDataFrame de barrios
barrios_cols = ["geometry"]
if "nombre" in gdf_barrios.columns:
    barrios_cols.append("nombre")
if "comuna" in gdf_barrios.columns:
    barrios_cols.append("comuna")

gdf = gdf.sjoin(gdf_barrios[barrios_cols], how="left", predicate="within")
gdf = gdf.drop(columns=["index_right"], errors="ignore")

# FIX: renombrar solo si las columnas existen tras el join
if "nombre" in gdf.columns:
    gdf = gdf.rename(columns={"nombre": "Barrio"})
if "comuna" in gdf.columns:
    gdf = gdf.rename(columns={"comuna": "Comuna"})

# ─── SUBTE ────────────────────────────────────────────────────────────────────

print("🚇 Subte...")
gdf["Dist_Subte_m"]  = distancia_al_mas_cercano(gdf, gdfs["subtes"])

# FIX: detectar la columna de nombre de estación (puede variar entre versiones del dataset)
subtes_gdf = gdfs["subtes"]
estacion_col = next((c for c in subtes_gdf.columns if "estacion" in c.lower() or "nombre" in c.lower()), None)
linea_col    = next((c for c in subtes_gdf.columns if "linea" in c.lower() or "line" in c.lower()), None)

if estacion_col:
    gdf["Subte_cercano"] = nombre_mas_cercano(gdf, subtes_gdf, estacion_col)
else:
    print("   ⚠️  No se encontró columna de nombre de estación en el dataset de subtes")

if linea_col:
    gdf["Linea_subte"] = nombre_mas_cercano(gdf, subtes_gdf, linea_col)
else:
    print("   ⚠️  No se encontró columna de línea en el dataset de subtes")

# ─── HOSPITAL ─────────────────────────────────────────────────────────────────

print("🏥 Hospitales...")
hospitales_gdf = gdfs["hospitales"]
gdf["Dist_Hospital_m"] = distancia_al_mas_cercano(gdf, hospitales_gdf)

# FIX: detectar columna de nombre del hospital
hosp_name_col = next((c for c in hospitales_gdf.columns if c.lower() in ("nam", "nombre", "name")), None)
if hosp_name_col:
    gdf["Hospital_cercano"] = nombre_mas_cercano(gdf, hospitales_gdf, hosp_name_col)
else:
    print("   ⚠️  No se encontró columna de nombre en hospitales")

# ─── OSM: COLEGIOS, AVENIDAS, COMISARÍAS, GIMNASIOS, SUPERMERCADOS ────────────

print("\n📥 Descargando POIs de OpenStreetMap...")

OSM_QUERY = """
[out:json][timeout:90];
area["name"="Ciudad Autónoma de Buenos Aires"]->.caba;
(
  node["amenity"="school"](area.caba);
  node["amenity"="police"](area.caba);
  node["leisure"="fitness_centre"](area.caba);
  node["amenity"="gym"](area.caba);
  node["shop"="supermarket"](area.caba);
  way["highway"="primary"](area.caba);
  way["highway"="secondary"](area.caba);
);
out body;
>;
out skel qt;
"""

# FIX: se eliminó "shop"="hardware" que estaba en el query original pero no se usaba

try:
    resp = requests.post("https://overpass-api.de/api/interpreter", data=OSM_QUERY, timeout=90)
    resp.raise_for_status()  # FIX: lanzar error si HTTP falla
    elements = resp.json().get("elements", [])
    print(f"   ✅ {len(elements)} elementos descargados de OSM")

    def make_gdf(amenity_key, amenity_val):
        pts = [Point(e["lon"], e["lat"]) for e in elements
               if e.get("type") == "node" and e.get("tags", {}).get(amenity_key) == amenity_val]
        # FIX: retornar GeoDataFrame vacío con CRS correcto aunque no haya puntos
        return gpd.GeoDataFrame(geometry=pts, crs="EPSG:4326") if pts else gpd.GeoDataFrame(geometry=gpd.GeoSeries([], crs="EPSG:4326"))

    gimnasio_pts = [
        Point(e["lon"], e["lat"]) for e in elements
        if e.get("type") == "node" and (
            e.get("tags", {}).get("leisure") == "fitness_centre" or
            e.get("tags", {}).get("amenity") == "gym"
        )
    ]

    osm = {
        "colegios":      make_gdf("amenity", "school"),
        "comisarias":    make_gdf("amenity", "police"),
        "gimnasios":     gpd.GeoDataFrame(geometry=gimnasio_pts, crs="EPSG:4326") if gimnasio_pts
                         else gpd.GeoDataFrame(geometry=gpd.GeoSeries([], crs="EPSG:4326")),
        "supermercados": make_gdf("shop", "supermarket"),
    }

    # ── Avenidas desde ways ────────────────────────────────────────────────────
    # FIX: construir el índice nodo→nombre de calle en un solo paso
    node_to_street = {}
    for e in elements:
        if e.get("type") == "way" and e.get("tags", {}).get("highway") in ("primary", "secondary"):
            street_name = e.get("tags", {}).get("name", "Avenida")
            for nd in e.get("nodes", []):
                node_to_street[nd] = street_name

    # FIX: usar dict para lookup O(1) en lugar de recorrer elements dos veces
    node_coords = {e["id"]: (e["lon"], e["lat"]) for e in elements if e.get("type") == "node"}

    av_geom   = []
    av_labels = []
    for node_id, street_name in node_to_street.items():
        if node_id in node_coords:
            lon, lat = node_coords[node_id]
            av_geom.append(Point(lon, lat))
            av_labels.append(street_name)

    osm["avenidas"] = gpd.GeoDataFrame({"nombre": av_labels, "geometry": av_geom}, crs="EPSG:4326") \
                      if av_geom else gpd.GeoDataFrame({"nombre": [], "geometry": gpd.GeoSeries([], crs="EPSG:4326")})

    for k, v in osm.items():
        print(f"   {k}: {len(v)} registros")

    print("\n🏫 Colegios...")
    gdf["Dist_Colegio_m"] = distancia_al_mas_cercano(gdf, osm["colegios"])
    gdf["Colegios_500m"]  = count_within(gdf, osm["colegios"], radius=500)

    print("👮 Comisarías...")
    gdf["Dist_Comisaria_m"] = distancia_al_mas_cercano(gdf, osm["comisarias"])

    print("🏋️  Gimnasios...")
    gdf["Dist_Gimnasio_m"] = distancia_al_mas_cercano(gdf, osm["gimnasios"])

    print("🛒 Supermercados...")
    gdf["Dist_Supermercado_m"] = distancia_al_mas_cercano(gdf, osm["supermercados"])
    gdf["Supermercados_500m"]  = count_within(gdf, osm["supermercados"], radius=500)

    print("🛣️  Avenidas...")
    if len(osm["avenidas"]) > 0:
        gdf["Dist_Avenida_m"]  = distancia_al_mas_cercano(gdf, osm["avenidas"])
        gdf["Avenida_cercana"] = nombre_mas_cercano(gdf, osm["avenidas"], "nombre")
    else:
        print("   ⚠️  No se encontraron avenidas en OSM")

except Exception as e:
    print(f"❌ Error OSM: {e}")

# ─── COLECTIVOS (paradas a 300m) ──────────────────────────────────────────────

print("\n🚌 Colectivos OSM...")

COL_STOP_QUERY = """
[out:json][timeout:90];
area["name"="Ciudad Autónoma de Buenos Aires"]->.caba;
(
  node["highway"="bus_stop"](area.caba);
);
out body;
"""

# FIX: se eliminó la primera request (relaciones de bus) que se descartaba de todos modos
try:
    resp3 = requests.post("https://overpass-api.de/api/interpreter", data=COL_STOP_QUERY, timeout=90)
    resp3.raise_for_status()
    stops = resp3.json().get("elements", [])

    # FIX: filtrar elementos sin lon/lat para evitar KeyError
    stops = [e for e in stops if "lon" in e and "lat" in e]

    gdf_stops = gpd.GeoDataFrame(
        {"linea": [e.get("tags", {}).get("ref", "") for e in stops]},
        geometry=[Point(e["lon"], e["lat"]) for e in stops],
        crs="EPSG:4326"
    ).to_crs("EPSG:22185")

    gdf["Paradas_colectivo_300m"] = gdf.geometry.apply(
        lambda geom: (gdf_stops.distance(geom) <= 300).sum()
    )
    print(f"   ✅ {len(gdf_stops)} paradas de colectivo")

except Exception as e:
    print(f"❌ Error colectivos: {e}")

# ─── GUARDAR ──────────────────────────────────────────────────────────────────

df_final = pd.DataFrame(gdf.drop(columns=["geometry"]))
df_final.to_csv("Argenprop_Enriched.tsv", sep="\t", index=False, encoding="utf-8-sig")
print(f"\n✅ Listo! {len(df_final.columns)} columnas → argenprop_enriched.tsv")

cols_preview = [c for c in ["Barrio", "Comuna", "Dist_Subte_m", "Subte_cercano", "Dist_Hospital_m", "Dist_Colegio_m"] if c in df_final.columns]
print(df_final[cols_preview].head())

📥 Descargando datasets del GCBA...
   ✅ barrios
   ✅ subtes
   ✅ hospitales
   ✅ universidades

🏘️  Barrio y comuna...
🚇 Subte...
🏥 Hospitales...

📥 Descargando POIs de OpenStreetMap...
❌ Error OSM: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter

🚌 Colectivos OSM...
   ✅ 6680 paradas de colectivo

✅ Listo! 59 columnas → argenprop_enriched.tsv
    Barrio  Comuna  Dist_Subte_m       Subte_cercano  Dist_Hospital_m
0  Palermo    14.0   1262.253345  R.SCALABRINI ORTIZ       559.868782
1  Palermo    14.0    973.273291        PLAZA ITALIA      1574.570048
2  Palermo    14.0    910.499103              BULNES       569.599031
3  Palermo    14.0    167.776133              BULNES       719.193645
5  Palermo    14.0    494.425060        PLAZA ITALIA      1539.466554


In [ ]:
df

In [ ]:
df = 